# OWID CO₂ per capita (OWID population only)
Uses only the Python standard library.

1. Drop aggregates (`OWID_WRL`, blank ISO).
2. Inner-join OWID `co2` (million tonnes) to **OWID** `population` on `iso_code` + `year`.
3. Per capita = `co2 * 1e6 / population` (tonnes).

Do **not** use World Bank `SP.POP.TOTL` as the denominator. WB population is a mid-year mix of UN WPP and national offices — a different vintage. Result grain is **country-year**.


In [ ]:
import csv
from pathlib import Path

root = Path(".")

def keep(iso, name=""):
    code = (iso or "").strip().upper()
    if len(code) != 3 or code.startswith("OWID_"):
        return False
    return str(name).strip().lower() not in {"world", "africa", "asia", "europe"}

co2 = {}
with (root / "data/samples/owid-co2-sample.csv").open() as fh:
    for row in csv.DictReader(fh):
        if not keep(row["iso_code"], row["country"]):
            continue
        co2[(row["iso_code"].upper(), int(row["year"]))] = float(row["co2"])

pop = {}
with (root / "data/samples/owid-population-sample.csv").open() as fh:
    for row in csv.DictReader(fh):
        if not keep(row["iso_code"], row["country"]):
            continue
        pop[(row["iso_code"].upper(), int(row["year"]))] = float(row["population"])

assert ("OWID_WRL", 2020) not in co2
joined = []
for key, mt in sorted(co2.items()):
    if key not in pop:
        continue
    joined.append((key[0], key[1], mt, pop[key], mt * 1e6 / pop[key]))
assert joined and len(joined) == len(set(r[:2] for r in joined))
print("iso year co2_mt population t_per_capita")
for row in joined:
    print(*row)
print(f"{len(joined)} country-year rows using OWID population only")
